# 09 - Reconstruction du pipeline du rapport de mars 2026

Reconstruction, à partir du texte du rapport (code dans `report_pipeline.py`), de la soumission principale :
LightGBM sur ~80 features, moyenne de 15 graines, puis mélange avec un vote momentum
p = 0,65 × modèle + 0,35 × (0,5·1[RET_1>0] + 0,3·1[PERF_3>0] + 0,2·1[PERF_5>0]).

Écarts avec l'original : pas d'Ichimoku, de Chandelier ni de target encoding de clusters ; 250 arbres fixés.
La reconstruction retrouve les chiffres du rapport sur toutes les dates (52,46 % pour le modèle seul contre 52,41 % annoncé,
52,07 % pour le mélange contre 52,07 % annoncé).

Compter 5 à 10 minutes (15 graines sur tout le train).

In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path

import utils
import report_pipeline as rp

X, y, Xt = utils.load_data()
up = (y > 0).to_numpy()

utils.load_journal().set_index('modele').loc[
    ['rapport2026_vote', 'rapport2026_blend', 'rapport2026_modele', 'benchmark_lgbm', 'final_blend'],
    ['recent_moy', 'toutes_moy']].round(4)

,recent_moy,toutes_moy
modele,,
rapport2026_vote,0.5131,0.5150
rapport2026_blend,0.5163,0.5207
rapport2026_modele,0.5195,0.5246
benchmark_lgbm,0.5184,0.5217
final_blend,0.5236,0.5278


Sur nos plis, dates récentes : le mélange du rapport fait 51,63 %, sous le benchmark (51,84 %) et sous son propre modèle seul
(51,95 %). Comme prévu, le vote momentum domine la décision et tire le score vers celui de la règle naïve.
Notre mélange final est à 52,36 %.

In [2]:
F, Ft = rp.report_features(X), rp.report_features(Xt)
bounds = rp.winsorize_fit(F)
F, Ft = rp.winsorize_apply(F, bounds), rp.winsorize_apply(Ft, bounds)

p_model = np.zeros(len(Xt))
for seed in range(15):
    booster = lgb.train({**rp.LGBM_PARAMS, 'seed': seed}, lgb.Dataset(F, label=up.astype(int)), num_boost_round=rp.N_ROUNDS)
    p_model += booster.predict(Ft) / 15
p_blend = (1 - rp.ALPHA) * p_model + rp.ALPHA * rp.momentum_vote(Xt)

sample = pd.read_csv(utils.find_data_dir() / 'sample_submission.csv', index_col='ROW_ID')
sub = pd.DataFrame({sample.columns[0]: (p_blend > .5).astype(int)}, index=Xt.index).loc[sample.index]
sub.index.name = 'ROW_ID'
Path('submissions').mkdir(exist_ok=True)
sub.to_csv('submissions/10_rapport_2026.csv')

vote_only = (rp.momentum_vote(Xt) > .5).astype(int)
print(sub.shape, '| part de 1 :', round(sub.iloc[:, 0].mean(), 3))
print("part des prédictions identiques au vote momentum seul :", round((sub.iloc[:, 0].to_numpy() == pd.Series(vote_only, index=Xt.index).loc[sample.index].to_numpy()).mean(), 3))

(31870, 1) | part de 1 : 0.537
part des prédictions identiques au vote momentum seul : 0.865


La dernière ligne mesure directement le problème relevé dans le rapport : la part des prédictions du mélange qui sont
identiques à celles du vote momentum seul. Le modèle ne change la décision que pour une petite minorité de lignes.